<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_03_gru_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_02 - MLP(Direct Multi-Step)**


## **Introducción**

**MLP (Direct Multi-step)**
   - Aplana $(29 \times 7)$ y predice $(29 \times 1)$.
   - Base neural simple y rápida para validar el pipeline.
   - Notebook: `stage_07_02_mlp_direct_multistep.ipynb`


**Output:** métricas y predicciones out-of-sample guardadas como artefactos para el **Stage_08**.

El MLP (Multi-Layer Perceptron) es una red neuronal feed-forward clásica, compuesta por:

- una capa de entrada  
- una o más capas ocultas  
- una capa de salida  

Cada capa realiza dos operaciones fundamentales:  
1) una combinación lineal de las entradas mediante pesos entrenables,  
2) una función de activación no lineal, que permite modelar relaciones complejas entre variables.

---

**Qué hace conceptualmente**

El MLP aprende una función que mapea las variables de entrada hacia el target:

$$ X → Y $$

Este aprendizaje se lleva a cabo ajustando los pesos del modelo para minimizar una función de error entre las predicciones generadas y los valores reales del objetivo.

A diferencia del modelo Naive, el MLP:

- sí entrena  
- ajusta parámetros a partir de los datos  
- puede capturar relaciones no lineales entre las variables de entrada y el target  

---

**Rol del MLP en el proyecto MNQ**

En el contexto de este proyecto:

- El MLP recibe una ventana fija de datos intradía, generalmente aplanada (flattened) o previamente agregada.  
- Produce una predicción directa del target, ya sea un vector de deltas futuros o una secuencia multi-step.  
- No modela explícitamente la dependencia temporal entre pasos consecutivos; en cambio, aprende patrones estadísticos contenidos en la ventana completa.

Por este motivo, el MLP se utiliza como:

- el primer modelo entrenable de referencia  
- un puente metodológico entre el baseline Naive y los modelos secuenciales más complejos (LSTM, GRU, Transformer)

En las siguientes secciones se detalla la definición precisa de entradas y salidas, la arquitectura utilizada y el procedimiento de entrenamiento y validación, siguiendo un esquema de evaluación basado exclusivamente en el conjunto VALID.

## **1. Imports + paths**

In [1]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

In [2]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:

#VENTANAS 60MIN
IN_WINDOW_TRAIN_60_Z = Path(os.environ.get("IN_WINDOW_TRAIN_60_Z", "data/windows/scaled/windows_train_60_z.npz"))
IN_WINDOW_VALID_60_Z = Path(os.environ.get("IN_WINDOW_VALID_60_Z", "data/windows/scaled/windows_valid_60_z.npz"))
IN_WINDOW_TEST_60_Z = Path(os.environ.get("IN_WINDOW_TEST_60_Z", "data/windows/scaled/windows_test_60_z.npz"))
IN_SCALER_60 = Path(os.environ.get("IN_SCALER_60", "data/windows/scaled/scaler_60.pkl"))

#VENTANAS 90MIN
IN_WINDOW_TRAIN_90_Z = Path(os.environ.get("IN_WINDOW_TRAIN_90_Z", "data/windows/scaled/windows_train_90_z.npz"))
IN_WINDOW_VALID_90_Z = Path(os.environ.get("IN_WINDOW_VALID_90_Z", "data/windows/scaled/windows_valid_90_z.npz"))
IN_WINDOW_TEST_90_Z = Path(os.environ.get("IN_WINDOW_TEST_90_Z", "data/windows/scaled/windows_test_90_z.npz"))
IN_SCALER_90 = Path(os.environ.get("IN_SCALER_90", "data/windows/scaled/scaler_90.pkl"))

#ARTIFACTS

# Summary del stage_03a (donde está delta_target_p70 por horizonte).
IN_TARGET_INVESTIGATION_SUMMARY = Path(os.environ.get("IN_TARGET_INVESTIGATION_SUMMARY", "reports/stage_03a_target_investigation_summary.json"))

# Summary del stage_06 (donde está window_size y n_features por horizonte).
IN_WINDOWS_SCALING_SUMMARY =Path(os.environ.get("IN_WINDOWS_SCALING_SUMMARY", "reports/stage_06_window_scaling_seq2seq_summary.json"))

#OUT_MODEL_METRICS = Path(os.environ.get("OUT_MODEL_METRICS", f"reports/stage_07__model_metrics.json"))

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


In [4]:
#PARA LA NOTEBOOK
IN_WINDOW_TRAIN_60_Z = DRIVE_DIR / IN_WINDOW_TRAIN_60_Z
IN_WINDOW_VALID_60_Z = DRIVE_DIR / IN_WINDOW_VALID_60_Z
IN_WINDOW_TEST_60_Z = DRIVE_DIR / IN_WINDOW_TEST_60_Z
IN_SCALER_60 = DRIVE_DIR / IN_SCALER_60

IN_WINDOW_TRAIN_90_Z = DRIVE_DIR / IN_WINDOW_TRAIN_90_Z
IN_WINDOW_VALID_90_Z=DRIVE_DIR / IN_WINDOW_VALID_90_Z
IN_WINDOW_TEST_90_Z = DRIVE_DIR / IN_WINDOW_TEST_90_Z
IN_SCALER_90 = DRIVE_DIR / IN_SCALER_90

IN_WINDOWS_SCALING_SUMMARY = DRIVE_DIR / IN_WINDOWS_SCALING_SUMMARY
IN_TARGET_INVESTIGATION_SUMMARY = DRIVE_DIR / IN_TARGET_INVESTIGATION_SUMMARY

## **2. Reproducibilidad**

In [5]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **3. Configuración**

In [6]:
def _read_json(path: Path) -> Dict[str, Any]:
    """Lee un JSON y devuelve un dict Python (con validación básica de existencia)."""
    # Verifica que el archivo exista antes de abrirlo.
    if not path.exists():
        # Si no existe, corta la ejecución con un error claro.
        raise FileNotFoundError(f"No existe el JSON: {path}")
    # Abre el archivo en modo lectura, asegurando UTF-8.
    with path.open("r", encoding="utf-8") as f:
        # Parsea el contenido JSON y lo devuelve como dict.
        return json.load(f)

In [7]:
# Config final para entrenar (modelo, horizonte).
@dataclass(frozen=True)
class StageConfig:
    # Horizonte (60 o 90).
    horizon: int
    # Largo de ventana (timesteps) desde stage_06.
    seq_len: int
    # Cantidad de features desde stage_06 para ese horizonte.
    n_features: int
    # Nombres de features (orden exacto) para ese horizonte.
    feature_names: List[str]
    # Nombre del target para ese horizonte.
    target_name: List[str]
    # Umbral mínimo económico (DELTA_BASE).
    delta_base: float
    # Umbral de oportunidad (DELTA_OP) leído del stage_03a.
    delta_op: float

In [8]:
# Construye un StageConfig leyendo ambos reports.
def load_state_from_reports(
    horizon: int,
    #model_name: str,
    *,
    in_windows_scaling_summary: Path = IN_WINDOWS_SCALING_SUMMARY,
    in_target_investigation_summary: Path = IN_TARGET_INVESTIGATION_SUMMARY,
) -> StageConfig:
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Carga JSON del stage_06.
    w = _read_json(in_windows_scaling_summary)

    # Carga JSON del stage_03a.
    t = _read_json(in_target_investigation_summary)

    # Lee window_size global (SEQ_LEN).
    seq_len = int(w["details"]["config"]["window_size"])

    # Selecciona dataset del horizonte (ojo: "60" o "90" como string).
    ds = w["details"]["datasets"][str(horizon)]

    # Lee n_features del horizonte.
    n_features = int(ds["n_features"])

    # Lee feature_names del horizonte (aquí se refleja el 1 feature distinto).
    feature_names = list(ds["feature_names"])

    # Lee target del horizonte.
    target_name = ds["target"]

    # Valida consistencia.
    if len(feature_names) != n_features:
        raise ValueError("Inconsistencia entre n_features y feature_names")

    # Define la key de delta_base_med del stage_03a.
    target_base = f"h{horizon}_delta_base_med"
    delta_base = float(t["metrics"][target_base])

    # Define la key de delta_target_p70 del stage_03a.
    target_op = f"h{horizon}_delta_target_p70"
    # Lee DELTA_OP para ese horizonte.
    delta_op = float(t["metrics"][target_op])

    # Devuelve la config lista para entrenar.
    return StageConfig(
        horizon=horizon,
        seq_len=seq_len,
        n_features=n_features,
        feature_names=feature_names,
        target_name=target_name,
        delta_base=float(delta_base),
        delta_op=float(delta_op),
            )


In [9]:
states_h60 = load_state_from_reports(horizon=60)
states_h60

StageConfig(horizon=60, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_60'], target_name='delta_pts_60', delta_base=52.12, delta_op=84.18)

In [10]:
states_h90 = load_state_from_reports(horizon=90)
states_h90

StageConfig(horizon=90, seq_len=29, n_features=7, feature_names=['open', 'high', 'low', 'close', 'ema_60', 'mom_10_struct', 'roc_30'], target_name='delta_pts_90', delta_base=60.75, delta_op=97.22)


## **4. Importar métricas comunes desde .py**

In [11]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2seq_metrics import compute_seq2seq_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [12]:
print(compute_seq2seq_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2seq.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    y_pred : np.ndarray
        Valores predichos con shape (n_samples, seq_len) o (n_samples, seq_len, 1)
    compute_r2 : bool
        Si True, calcula R² sobre la secuencia completa concatenada.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 (y_true_last==0 o y_pred_last==0)
        al calcular DA_last. Esto evita ambigüedad en la dirección.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales y diagnóstico por paso.
    


## **5. Carga de data windows**

In [13]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [14]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [15]:
# --------------------------------------------------
# Carga completa: train/valid/test + scaler por horizonte
# --------------------------------------------------
def load_windows_and_scaler_for_horizon(horizon: int) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler para un horizonte dado (60 o 90).

    Retorna un dict:
    {
      "horizon": 60,
      "paths": {...},
      "scaler": <StandardScaler>,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }
    """
    # Valida horizonte permitido.
    if horizon not in (60, 90):
        raise ValueError("horizon debe ser 60 o 90")

    # Selecciona paths según horizonte.
    if horizon == 60:
        train_path = IN_WINDOW_TRAIN_60_Z
        valid_path = IN_WINDOW_VALID_60_Z
        test_path  = IN_WINDOW_TEST_60_Z
        scaler_path = IN_SCALER_60
    else:
        train_path = IN_WINDOW_TRAIN_90_Z
        valid_path = IN_WINDOW_VALID_90_Z
        test_path  = IN_WINDOW_TEST_90_Z
        scaler_path = IN_SCALER_90

    # Carga ventanas.
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    # Carga scaler.
    scaler = load_scaler(scaler_path)

    # Retorna todo empaquetado.
    return {
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [16]:
# --------------------------------------------------
# Carga efectiva: H60 y H90 (dos datasets distintos)
# --------------------------------------------------

# Carga todo para 60 min.
bundle_60 = load_windows_and_scaler_for_horizon(60)

# Carga todo para 90 min.
bundle_90 = load_windows_and_scaler_for_horizon(90)


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [17]:
# --------------------------------------------------
# Verificación rápida
# --------------------------------------------------

# Shapes H60.
print("H60 Train:", bundle_60["train"]["X"].shape, bundle_60["train"]["y"].shape)
print("H60 Valid:", bundle_60["valid"]["X"].shape, bundle_60["valid"]["y"].shape)
print("H60 Test :", bundle_60["test"]["X"].shape,  bundle_60["test"]["y"].shape)

# Shapes H90.
print("H90 Train:", bundle_90["train"]["X"].shape, bundle_90["train"]["y"].shape)
print("H90 Valid:", bundle_90["valid"]["X"].shape, bundle_90["valid"]["y"].shape)
print("H90 Test :", bundle_90["test"]["X"].shape,  bundle_90["test"]["y"].shape)

# Información útil (scaler).
print("Scaler H60:", type(bundle_60["scaler"]).__name__)
print("Scaler H90:", type(bundle_90["scaler"]).__name__)

H60 Train: (912, 29, 7) (912, 29)
H60 Valid: (195, 29, 7) (195, 29)
H60 Test : (196, 29, 7) (196, 29)
H90 Train: (912, 29, 7) (912, 29)
H90 Valid: (195, 29, 7) (195, 29)
H90 Test : (196, 29, 7) (196, 29)
Scaler H60: StandardScaler
Scaler H90: StandardScaler


NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **6. Sanity Check**

In [18]:
def sanity_check(
    X: np.ndarray,                 # Tensores de entrada: (n_samples, seq_len, n_features)
    y: np.ndarray,                 # Targets: (n_samples, seq_len) o (n_samples, seq_len, 1)
    name: str,                     # Nombre lógico del split (ej: "train_h60", "valid_h90")
    *,
    expected_seq_len: int,         # Largo de secuencia esperado (ej: 29)
    expected_n_features: int,      # Número de features esperado (ej: 8)
) -> None:
    # --------------------------------------------------
    # Chequeos de dimensionalidad
    # --------------------------------------------------

    # X debe ser estrictamente 3D: (muestras, tiempo, features)
    assert X.ndim == 3, (
        f"{name}: X debe ser 3D (n, seq, feat)"
    )

    # y puede ser 2D (n, seq) o 3D (n, seq, 1)
    assert y.ndim in (2, 3), (
        f"{name}: y debe ser 2D o 3D (n, seq) o (n, seq, 1)"
    )

    # --------------------------------------------------
    # Chequeos de consistencia temporal y estructural
    # --------------------------------------------------

    # Verifica que el largo temporal de X coincida con el esperado
    assert X.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en X: {X.shape[1]} != {expected_seq_len}"
    )

    # Verifica que la cantidad de features en X sea la esperada
    assert X.shape[2] == expected_n_features, (
        f"{name}: n_features inesperado en X: {X.shape[2]} != {expected_n_features}"
    )

    # Verifica que y tenga el mismo largo temporal que X
    assert y.shape[1] == expected_seq_len, (
        f"{name}: seq_len inesperado en y: {y.shape[1]} != {expected_seq_len}"
    )

    # --------------------------------------------------
    # Chequeos numéricos (sanidad de valores)
    # --------------------------------------------------

    # Asegura que X no contenga NaN ni infinitos
    assert np.isfinite(X).all(), (
        f"{name}: X contiene NaN/inf"
    )

    # Asegura que y no contenga NaN ni infinitos
    assert np.isfinite(y).all(), (
        f"{name}: y contiene NaN/inf"
    )

In [19]:
from __future__ import annotations

from typing import Any, Dict
import numpy as np


def run_sanity_checks_for_bundle(bundle: Dict[str, Any], *, tag: str) -> None:
    """
    Ejecuta sanity_check para train/valid/test usando únicamente variables locales.

    Espera un bundle con estructura:
    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
      ...
    }
    """
    # Extrae arrays localmente (no crea globals)
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]

    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]

    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    # Toma expected_* desde TRAIN (consistencia)
    expected_seq_len = int(X_tr.shape[1])
    expected_n_features = int(X_tr.shape[2])

    # Ejecuta sanity checks por split
    sanity_check(X_tr, y_tr, f"train_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_va, y_va, f"valid_{tag}", expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)
    sanity_check(X_te, y_te, f"test_{tag}",  expected_seq_len=expected_seq_len, expected_n_features=expected_n_features)

    # Mensaje de OK por bundle/horizonte
    h = bundle.get("horizon", "NA")
    print(f"OK {tag} (h={h}) | seq_len={expected_seq_len} | n_features={expected_n_features}")

    # Opcional: borra referencias locales explícitamente (no es estrictamente necesario)
    del X_tr, y_tr, X_va, y_va, X_te, y_te


def run_sanity_checks_all_horizons(bundle_60: Dict[str, Any], bundle_90: Dict[str, Any]) -> None:
    """Corre sanity checks para ambos horizontes."""
    run_sanity_checks_for_bundle(bundle_60, tag="h60")
    run_sanity_checks_for_bundle(bundle_90, tag="h90")

In [20]:
run_sanity_checks_all_horizons(bundle_60, bundle_90)

OK h60 (h=60) | seq_len=29 | n_features=7
OK h90 (h=90) | seq_len=29 | n_features=7


## **8. Definición del modelo — placeholder**

In [36]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

En un MLP Direct Multi-step:

- El riesgo principal es el sobreajuste por capacidad
(muchos parámetros, sin estructura temporal explícita).
- Por eso, la regularización no es opcional, es parte del modelo.

La jerarquía correcta es:

- L2 (weight decay) → mecanismo principal
- Early stopping en VALID → control dinámico
- Arquitectura limitada → prevención estructural
- Dropout leve → solo si hace falta

Tal como indica Jansen:
- Modelo flexible que requiere regularización explícita para generalizar.

### **8.1. Definición de modelo con regularización**

In [37]:
def build_mlp_direct_multistep(
    window_len: int = 29,
    n_features: int = 7,
    horizon_len: int = 29,
    hidden_units=(128, 64),      # arquitectura contenida
    dropout=0.10,                # leve
    l2_reg=1e-4,                 # L2 principal
    lr=1e-3,
) -> keras.Model:

    inputs = keras.Input(shape=(window_len, n_features), name="X_window")
    x = layers.Flatten(name="flatten")(inputs)

    for i, u in enumerate(hidden_units, start=1):
        x = layers.Dense(
            u,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_reg),
            name=f"dense_{i}",
        )(x)
        x = layers.Dropout(dropout, name=f"dropout_{i}")(x)

    outputs = layers.Dense(
        horizon_len,
        activation="linear",
        kernel_regularizer=regularizers.l2(l2_reg),
        name="y_hat",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="MLP_Direct_MultiStep")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=[keras.metrics.MAE],
    )
    return model


In [38]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)

### **8.2. Construcción de modelo desde bundles**

In [39]:
def build_model_for_bundle(bundle_h):
    X_train = bundle_h["train"]["X"]  # (N, window_len, n_features)
    y_train = bundle_h["train"]["y"]  # (N, horizon_len) o (N, horizon_len, 1)

    window_len = int(X_train.shape[1])
    n_features = int(X_train.shape[2])

    # Normaliza y_train a 2D: (N, horizon_len)
    if y_train.ndim == 3:
        if y_train.shape[-1] != 1:
            raise ValueError(f"Se esperaba y_train con último dim=1, got {y_train.shape}")
        y_train_2d = y_train[..., 0]
    elif y_train.ndim == 2:
        y_train_2d = y_train
    else:
        raise ValueError(f"Forma inesperada para y_train: {y_train.shape}")

    horizon_len = int(y_train_2d.shape[1])

    model = build_mlp_direct_multistep(
        window_len=window_len,
        n_features=n_features,
        horizon_len=horizon_len,
    )
    return model

### **8.3. Entrenamiento MLP (train + valid, sin test)**

In [40]:
import numpy as np

def to_2d_y(y):
    # (N, H, 1) -> (N, H) ; (N, H) queda igual
    if y.ndim == 3 and y.shape[-1] == 1:
        return y.reshape(y.shape[0], y.shape[1])
    if y.ndim == 2:
        return y
    raise ValueError(f"Forma inesperada para y: {y.shape}")

In [41]:
def fit_mlp_on_bundle_valid_only(
    bundle_h,
    model: keras.Model,
    *,
    epochs: int = 200,
    batch_size: int = 256,
    patience: int = 10,
    verbose: int = 1,
):
    X_train = bundle_h["train"]["X"]
    y_train = to_2d_y(bundle_h["train"]["y"])

    X_valid = bundle_h["valid"]["X"]
    y_valid = to_2d_y(bundle_h["valid"]["y"])

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_valid, y_valid),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=verbose,
    )
    return history

### **8.4. Predicción y evaluación (VALID ONLY)**

In [42]:
def predict_mlp_2d(bundle_h, model, split="valid"):
    X = bundle_h[split]["X"]
    y_true = to_2d_y(bundle_h[split]["y"])
    y_pred = model.predict(X, verbose=0)  # (N, H)
    return y_true, y_pred

In [43]:
import pandas as pd

def metrics_to_df(metrics: dict, *, model: str, split: str, horizon: int) -> pd.DataFrame:
    return pd.DataFrame([{
        "model": model,
        "split": split,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA_last": metrics.get("DA_last"),
    }])

### **8.5. Flujo completo para H60 y H90**

In [44]:
# -------------------------
# H60
# -------------------------
model_60 = build_model_for_bundle(bundle_60)
fit_mlp_on_bundle_valid_only(bundle_60, model_60, epochs=200, batch_size=256, patience=10)

y_true_60, y_pred_60 = predict_mlp_2d(bundle_60, model_60, split="valid")
ml_valid_60 = compute_seq2seq_metrics(y_true_60, y_pred_60, compute_r2=True)

Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - loss: 2903.0850 - mean_absolute_error: 36.9337 - val_loss: 2504.6812 - val_mean_absolute_error: 36.4071
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 2667.3923 - mean_absolute_error: 36.1306 - val_loss: 2496.9299 - val_mean_absolute_error: 36.3675
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 2830.3284 - mean_absolute_error: 37.0849 - val_loss: 2489.0000 - val_mean_absolute_error: 36.3244
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 2715.2803 - mean_absolute_error: 36.6612 - val_loss: 2479.5786 - val_mean_absolute_error: 36.2757
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 2838.6904 - mean_absolute_error: 37.3170 - val_loss: 2467.8130 - val_mean_absolute_error: 36.2079
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 2857.4155 - mean_absolute_error: 36.5144 - val_loss: 2454.8848 - val_mean_absolute_error: 36.1405
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 2724.1

In [45]:
# -------------------------
# H90
# -------------------------
model_90 = build_model_for_bundle(bundle_90)
fit_mlp_on_bundle_valid_only(bundle_90, model_90, epochs=200, batch_size=256, patience=10)

y_true_90, y_pred_90 = predict_mlp_2d(bundle_90, model_90, split="valid")
ml_valid_90 = compute_seq2seq_metrics(y_true_90, y_pred_90, compute_r2=True)



Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - loss: 6051.4268 - mean_absolute_error: 58.1443 - val_loss: 5023.8491 - val_mean_absolute_error: 53.3949
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 6317.3809 - mean_absolute_error: 59.8540 - val_loss: 5018.4619 - val_mean_absolute_error: 53.3775
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 5707.5640 - mean_absolute_error: 57.6831 - val_loss: 5013.1079 - val_mean_absolute_error: 53.3565
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 6055.6050 - mean_absolute_error: 59.1280 - val_loss: 5005.4824 - val_mean_absolute_error: 53.3301
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 6171.5327 - mean_absolute_error: 59.1844 - val_loss: 4997.5342 - val_mean_absolute_error: 53.3013
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 5994.6538 - mean_absolute_error: 58.5752 - val_loss: 4987.0928 - val_mean_absolute_error: 53.2582
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 5809.0

,model,split,horizon_min,MAE,RMSE,R2,DA_last
0,mlp,valid,60,30.137121,39.786953,0.370110,0.484536
1,mlp,valid,90,50.746009,65.093589,0.157363,0.484536


## **9. Métricas ML**

In [47]:
# Tabla resumen
df_valid_60 = metrics_to_df(ml_valid_60, model="mlp", split="valid", horizon=60)
df_valid_90 = metrics_to_df(ml_valid_90, model="mlp", split="valid", horizon=90)

pd.concat([df_valid_60, df_valid_90], ignore_index=True)


,model,split,horizon_min,MAE,RMSE,R2,DA_last
0,mlp,valid,60,30.137121,39.786953,0.370110,0.484536
1,mlp,valid,90,50.746009,65.093589,0.157363,0.484536


## **11. Guardar artefactos para Stage_08**

In [48]:
def save_json(obj: Dict[str, Any], path: Path) -> None:
    """Guarda un diccionario como JSON, creando directorios si es necesario."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [49]:
from pathlib import Path
from typing import Any, Dict
import numpy as np

def save_evaluation_artifacts(
    *,
    out_dir: Path,
    model_name: str,
    horizon: int,
    ml_valid: Dict[str, Any],
    y_valid: np.ndarray | None = None,
    y_pred_valid: np.ndarray | None = None,
    save_preds: bool = True,
) -> None:
    """
    Guarda SOLO artefactos de VALID (coherente con el esquema del libro):
      - metrics_ml_valid.json
      - pred_valid.npz (opcional)

    No guarda nada de TEST.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # Métricas ML (VALID)
    # -------------------------
    save_json(
        {
            "model": model_name,
            "horizon_min": horizon,
            "split": "valid",
            **ml_valid,
        },
        out_dir / "metrics_ml_valid.json",
    )

    # -------------------------
    # Predicciones OOS (VALID) - opcional
    # -------------------------
    if save_preds:
        if y_valid is None or y_pred_valid is None:
            raise ValueError("Si save_preds=True, debe pasar y_valid y y_pred_valid.")
        np.savez_compressed(
            out_dir / "pred_valid.npz",
            y_true=np.asarray(y_valid),
            y_pred=np.asarray(y_pred_valid),
        )

    print(f"OK - artefactos guardados en: {out_dir}")

In [50]:
OUT_DIR_60 = Path("artifacts/02_mlp/h60")
OUT_DIR_60 = DRIVE_DIR / OUT_DIR_60 #Solo para la notebook

# y_true_60, y_pred_60 deben venir de predict_mlp_2d(..., split="valid")
save_evaluation_artifacts(
    out_dir=OUT_DIR_60,
    model_name="mlp_h60",
    horizon=60,
    ml_valid=ml_valid_60,
    y_valid=y_true_60,        # ← 2D seguro
    y_pred_valid=y_pred_60,   # ← 2D
    save_preds=False,
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/02_mlp/h60


In [51]:
OUT_DIR_90 = Path("artifacts/02_mlp/h90")
OUT_DIR_90 = DRIVE_DIR / OUT_DIR_90 #Solo para la notebook

save_evaluation_artifacts(
    out_dir=OUT_DIR_90,
    model_name="mlp_h90",
    horizon=90,
    ml_valid=ml_valid_90,
    y_valid=y_true_90,
    y_pred_valid=y_pred_90,
    save_preds=False,
)

OK - artefactos guardados en: /content/drive/MyDrive/neural_profit/artifacts/02_mlp/h90


## **12. Resumen**

- **Métricas ML (test):** MAE, RMSE, DA_last, R2
- **Métricas económicas como filtro (test):** Precision, Opportunity_Recall, Coverage
- Artefactos guardados en `reports/stage_07/h{H}/<model_name>/`

El MLP mejora claramente al Naive en ambos horizontes, capturando señal predictiva real. El desempeño es más sólido en H=60 (menor error y R² ≈ 0.37), mientras que en H=90 la señal es más débil pero aún positiva. La DA_last ≈ 0.48 indica que la dirección sigue siendo un desafío, consistente con el aumento de horizonte.